In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [75]:
X_train = pd.read_csv('../../../data/processed/for_linear_model/GK/predictors_train_filtered.csv')
X_test = pd.read_csv('../../../data/processed/for_linear_model/GK/predictors_test_filtered.csv')
y_train = pd.read_csv('../../../data/processed/for_linear_model/GK/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('../../../data/processed/for_linear_model/GK/target_test_filtered_logariphmed.csv')

In [6]:
train_players = pd.read_csv('../../../data/processed/for_linear_model/GK/train_players_names.csv')
test_players = pd.read_csv('../../../data/processed/for_linear_model/GK/test_players_names.csv')

In [8]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'GA', 'GA90', 'SoTA', 'Saves',
       'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt', 'PKA', 'PKsv', 'PKm',
       'PKsv%', 'league_ES1', 'league_FR1', 'league_GB1', 'league_IT1',
       'league_L1', 'league_NL1', 'league_PO1', 'league_RU1'],
      dtype='object')

## Регрессор

In [79]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

## Предсказанные значения

In [80]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

## Метрики

In [81]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

## Остатки

In [82]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

## Коэффициенты

In [83]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

# Все переменные

In [14]:
coeffs_df

,coeff
Min,-15.324961
90s,13.872235
SoTA,5.991010
Saves,-4.576390
GA,-2.041567
W,1.036480
Starts,0.809669
Age,-0.687525
league_GB1,0.661890
league_IT1,0.478799


In [15]:
metrics

,MAE,RMSE,MAPE,R2
test,2.526755,4.721005,0.67345,0.487875
train,2.627833,5.155999,0.54584,0.633245


# Потенциальные переменные для добавления и удаления

## Текущие переменные

In [74]:
base_cols = ['W', '90s', 'PKsv%', 'GA90', 'Age', 'CS%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

## Для добавления:

In [76]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,2.290762,0.660419,0.561213
MP,2.292114,0.804624,0.581363
Starts,2.281196,0.811967,0.580643
Min,2.310482,0.802534,0.589511
GA,2.289883,0.808664,0.582158
SoTA,2.287909,0.805450,0.578962
Saves,2.292280,0.806435,0.580961
Save%,2.297227,0.813479,0.581205
D,2.264819,0.808106,0.570762
L,2.258407,0.809874,0.569196


## Для удаления:

In [77]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,2.290762,0.660419,0.561213
W,2.286652,0.919916,0.420584
90s,2.376186,0.802093,0.577281
PKsv%,2.186760,0.798569,0.564861
CS%,2.295316,0.804931,0.581886


# Модель 1

In [19]:
X_train = X_train[['W', 'CS', '90s', 'GA90', 'Age', 'D', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['W', 'CS', '90s', 'GA90', 'Age', 'D', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [25]:
coeffs_df

,coeff
Age,-0.673402
league_GB1,0.659165
W,0.625911
league_IT1,0.483233
league_ES1,0.442786
league_L1,0.427698
league_FR1,0.410369
GA90,-0.215938
90s,0.189621
league_PO1,0.134140


In [26]:
metrics

,MAE,RMSE,MAPE,R2
test,2.167138,4.289290,0.631043,0.577255
train,2.832055,5.440427,0.571497,0.591665


In [29]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
56,Bart Verbruggen,46.151589,30.00
126,David Raya,44.687675,40.00
81,Robert Sanchez,40.215760,20.00
82,Mile Svilar,36.605173,25.00
17,Marco Carnesecchi,30.909874,25.00
169,Lucas Chevalier,29.409858,30.00
51,Anatolii Trubin,25.663555,28.00
137,Gianluigi Donnarumma,19.239964,40.00
122,Djordje Petrovic,16.036381,20.00
199,Guillaume Restes,15.907115,20.00


In [30]:
comp_leftovers_test.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
25,Alex Meret,25.448597,18.00
36,Marcin Bulka,20.991325,20.00
49,Matz Sels,20.802052,7.00
41,Alisson,20.774830,20.00
10,Ederson,19.568675,20.00
30,Dean Henderson,19.482980,20.00
15,Luiz Lucio Reis Junior,10.435337,10.00
40,Nick Pope,9.729860,8.00
4,David de Gea,8.341541,5.00
18,Philipp Kohn,7.475950,5.00


# Модель 2

In [35]:
X_train = X_train[['W', 'PKsv%', '90s', 'GA90', 'Age', 'D', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['W', 'PKsv%', '90s', 'GA90', 'Age', 'D', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [41]:
coeffs_df

,coeff
Age,-0.674138
league_GB1,0.666870
W,0.622551
league_IT1,0.490457
league_ES1,0.452531
league_L1,0.426264
league_FR1,0.416267
GA90,-0.208743
league_PO1,0.144321
90s,0.142297


In [42]:
metrics

,MAE,RMSE,MAPE,R2
test,2.271014,4.372885,0.672511,0.560617
train,2.767798,5.374240,0.567095,0.601540


In [48]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
56,Bart Verbruggen,43.076504,30.00
126,David Raya,42.617379,40.00
81,Robert Sanchez,40.542227,20.00
82,Mile Svilar,35.710253,25.00
17,Marco Carnesecchi,29.894185,25.00
169,Lucas Chevalier,28.058799,30.00
51,Anatolii Trubin,27.227531,28.00
137,Gianluigi Donnarumma,18.216115,40.00
199,Guillaume Restes,17.236077,20.00
153,Diogo Costa,16.201395,40.00


In [44]:
comp_leftovers_test

,Player,Value_pred,Value
0,Timon Wellenreuther,5.617790,6.00
1,Ilya Lantratov,1.492195,2.50
2,Unai Simon,7.605972,28.00
3,Christian Walton,1.810015,1.00
4,David de Gea,9.330813,5.00
5,Mads Hermansen,6.910078,15.00
6,Kaique,3.234330,0.60
7,Kaua Santos,5.889263,2.50
8,Ivan Lomaev,0.710235,1.50
9,Martin Delavallee,3.030339,1.50


# Модель 3

In [51]:
X_train = X_train[['W', 'PKsv%', 'GA90', 'Age', 'CS', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['W', 'PKsv%', 'GA90', 'Age', 'CS', 'CS%', 'Save%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [57]:
coeffs_df

,coeff
W,0.700690
Age,-0.675922
league_GB1,0.669110
league_IT1,0.496733
league_ES1,0.457510
league_L1,0.424145
league_FR1,0.423984
GA90,-0.171028
league_PO1,0.150236
league_NL1,0.080353


In [58]:
metrics

,MAE,RMSE,MAPE,R2
test,2.375219,4.504631,0.701419,0.533743
train,2.768157,5.414380,0.565917,0.595565


In [61]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
126,David Raya,44.779704,40.00
56,Bart Verbruggen,43.854366,30.00
81,Robert Sanchez,41.820546,20.00
82,Mile Svilar,35.869691,25.00
17,Marco Carnesecchi,30.003500,25.00
51,Anatolii Trubin,28.882270,28.00
169,Lucas Chevalier,28.451746,30.00
137,Gianluigi Donnarumma,19.688393,40.00
199,Guillaume Restes,16.813557,20.00
153,Diogo Costa,16.698517,40.00


In [60]:
comp_leftovers_test

,Player,Value_pred,Value
0,Timon Wellenreuther,6.225886,6.00
1,Ilya Lantratov,1.509380,2.50
2,Unai Simon,7.961154,28.00
3,Christian Walton,1.981683,1.00
4,David de Gea,9.314734,5.00
5,Mads Hermansen,6.489912,15.00
6,Kaique,3.445405,0.60
7,Kaua Santos,6.029480,2.50
8,Ivan Lomaev,0.722780,1.50
9,Martin Delavallee,3.064768,1.50


# Модель 4

In [66]:
X_train = X_train[['W', '90s', 'PKsv%', 'GA90', 'Age', 'CS%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['W', '90s', 'PKsv%', 'GA90', 'Age', 'CS%' ,'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [72]:
coeffs_df

,coeff
Age,-0.674760
league_GB1,0.670090
W,0.637754
league_IT1,0.488408
league_ES1,0.452761
league_L1,0.425379
league_FR1,0.424756
GA90,-0.212306
league_PO1,0.146867
90s,0.088225


In [73]:
metrics

,MAE,RMSE,MAPE,R2
test,2.290762,4.369917,0.660419,0.561213
train,2.767814,5.409834,0.566300,0.596244


# Модель 5

In [78]:
X_train = X_train[['90s', 'PKsv%', 'GA90', 'Age', 'CS%', 'Save%', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['90s', 'PKsv%', 'GA90', 'Age', 'CS%', 'Save%', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [84]:
coeffs_df

,coeff
league_GB1,0.685395
Age,-0.626589
90s,0.547705
GA90,-0.459792
league_FR1,0.457386
league_L1,0.455685
league_IT1,0.444537
league_ES1,0.442449
league_PO1,0.126352
Save%,-0.072955


In [85]:
metrics

,MAE,RMSE,MAPE,R2
test,2.333813,4.004961,0.593402,0.631444
train,2.970635,5.916928,0.656951,0.517004


In [86]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
56,Bart Verbruggen,40.620113,30.00
126,David Raya,36.397300,40.00
81,Robert Sanchez,28.686572,20.00
199,Guillaume Restes,24.408996,20.00
169,Lucas Chevalier,23.395670,30.00
82,Mile Svilar,21.204243,25.00
116,Jordan Pickford,21.001354,18.00
109,Andre Onana,19.998150,25.00
20,Zion Suzuki,17.195509,20.00
17,Marco Carnesecchi,16.339516,25.00


In [87]:
comp_leftovers_test

,Player,Value_pred,Value
0,Timon Wellenreuther,2.568340,6.00
1,Ilya Lantratov,0.809570,2.50
2,Unai Simon,8.599509,28.00
3,Christian Walton,1.285371,1.00
4,David de Gea,4.641305,5.00
5,Mads Hermansen,8.509222,15.00
6,Kaique,2.482537,0.60
7,Kaua Santos,4.766536,2.50
8,Ivan Lomaev,0.695744,1.50
9,Martin Delavallee,2.531262,1.50
